In [ ]:
# Importing the necessary libraries
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import aiohttp
import json
import asyncio
import async_lru

In [ ]:
@async_lru.alru_cache(maxsize=128)
async def get_data(year: int) -> pd.DataFrame:
    url = (

        "https://high-frequency-data.shriimpe.fr/api/data/price?start_date="
        + str(year)

        + "-01-01&end_date="
        + str(year)
        + "-12-31"
    )


    async with aiohttp.ClientSession() as session:
        async with session.get(url) as response:

            if response.status != 200:
                raise Exception(
                    f"Error fetching data from {url} (code {response.status})"
                )

            data = await response.text()


    data = json.loads(data)

    return pd.DataFrame(data).drop(columns=["volume"])



dfs = await asyncio.gather(*[get_data(year) for year in range(2009, 2024)])

In [ ]:
df = pd.concat(dfs)
df["date_time"] = pd.to_datetime(df["date_time"])
df.set_index("date_time", inplace=True)
df = df[df["price"] > 28]
df.head()

In [ ]:
noises = {}
for year in range(2009, 2024):
    for month in range(1, 13):
        for day in range(1, 32):
            try:
                date = datetime(year, month, day)
                tmp_df = df.loc[date : date + timedelta(days=1)]
                returns = tmp_df["price"].pct_change().dropna()
                variance = returns.var()
                noises[date] = variance
            except ValueError:
                pass

plt.figure(figsize=(20, 10))
plt.title("Microstructure noise")
plt.plot(noises.keys(), noises.values())
plt.show()